# Challenge 4 — Programming an Agent Workflow

A Gemini-only ADK workflow that searches for current information, critiques the draft, and refines the final answer.

In [ ]:
%pip install -q --upgrade google-adk

import vertexai
from google.adk.agents import LlmAgent, SequentialAgent
from google.adk.tools import google_search
from vertexai.preview import reasoning_engines

PROJECT_ID = "qwiklabs-gcp-02-9e12deb8c42f"
LOCATION = "us-central1"
MODEL_GEMINI = "gemini-2.5-flash"

vertexai.init(project=PROJECT_ID, location=LOCATION)
print("Setup complete.")

In [ ]:
search_agent = LlmAgent(
    name="search_agent",
    model=MODEL_GEMINI,
    description="Finds current, reliable information for an answer.",
    instruction=(
        "Use Google Search to research the user's question. Return a concise factual draft, "
        "including important caveats and source-grounded details."
    ),
    tools=[google_search],
    output_key="research_draft",
)

critique_agent = LlmAgent(
    name="critique_agent",
    model=MODEL_GEMINI,
    description="Checks a research draft for accuracy, completeness, and clarity.",
    instruction=(
        "Review the research draft in {research_draft}. Identify missing context, unsupported claims, "
        "safety concerns, and clarity improvements. Give specific revision instructions."
    ),
    output_key="critique_notes",
)

refine_agent = LlmAgent(
    name="refine_agent",
    model=MODEL_GEMINI,
    description="Produces a careful final answer from research and critique.",
    instruction=(
        "Write the final answer to the user's original question using the research draft in {research_draft} "
        "and the critique notes in {critique_notes}. Correct weaknesses, avoid unsupported claims, "
        "and be concise and useful. Do not mention this internal workflow."
    ),
    output_key="final_answer",
)

answer_team = SequentialAgent(
    name="answer_refinement_team",
    description="Researches, critiques, and refines answers in sequence.",
    sub_agents=[search_agent, critique_agent, refine_agent],
)

greeter_agent = LlmAgent(
    name="greeter_agent",
    model=MODEL_GEMINI,
    description="Receives user questions and sends them through the answer-refinement workflow.",
    instruction=(
        "Welcome the user briefly, then delegate every informational question to answer_refinement_team. "
        "Return the team's refined answer."
    ),
    sub_agents=[answer_team],
)

print("Created greeter, search, critique, refine, and sequential workflow agents.")

In [ ]:
# Workflow test. Event output demonstrates the greeter and every workflow specialist.
workflow_app = reasoning_engines.AdkApp(agent=greeter_agent)
TEST_QUESTION = "What should a family include in a hurricane preparedness kit?"

session = workflow_app.create_session(user_id="challenge-four-tester")
session_id = session["id"] if isinstance(session, dict) else session.id
print(f"=== Workflow test: {TEST_QUESTION} ===")

for event in workflow_app.stream_query(
    user_id="challenge-four-tester",
    session_id=session_id,
    message=TEST_QUESTION,
):
    author = event.get("author", "unknown") if isinstance(event, dict) else getattr(event, "author", "unknown")
    content = event.get("content") if isinstance(event, dict) else getattr(event, "content", None)
    parts = content.get("parts", []) if isinstance(content, dict) else getattr(content, "parts", [])
    text = " ".join(
        part.get("text", "") if isinstance(part, dict) else getattr(part, "text", "")
        for part in parts
    ).strip()
    if text:
        print(f"[{author}] {text}")

## Workflow architecture

```
User question
     ↓
greeter_agent
     ↓
answer_refinement_team (SequentialAgent)
   1. search_agent   → current research draft
   2. critique_agent → specific improvement notes
   3. refine_agent   → final answer
     ↓
Refined response returned to the user
```

The agents use ADK state keys (`research_draft`, `critique_notes`, and `final_answer`) to pass information through the workflow.